In [1]:
import time
import requests
import pandas as pd

PARAMS = {
    "filter": "4068",
    "region": "DE",
    "resolution": "hour",
}

BASE_URL = "https://www.smard.de/app/chart_data"


def request_data(url: str) -> dict:
    """Request data from URL and return parsed JSON."""
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


def build_timestamp_url(params: dict) -> str:
    """URL for the index of available weekly timestamps."""
    return (
        f"{BASE_URL}/{params['filter']}/{params['region']}"
        f"/index_{params['resolution']}.json"
    )


def build_timeseries_url(params: dict, timestamp: int) -> str:
    """URL for one weekly data chunk starting at 'timestamp'."""
    return (
        f"{BASE_URL}/{params['filter']}/{params['region']}"
        f"/{params['filter']}_{params['region']}_{params['resolution']}_{timestamp}.json"
    )


# Checkout which timestamps are available
timestamps = request_data(build_timestamp_url(PARAMS))["timestamps"]

# For every timestamp-chunk get the data and extend
series: list[list] = []
for i, timestamp in enumerate(timestamps):
    chunk = request_data(build_timeseries_url(PARAMS, timestamp))
    series.extend(chunk["series"])
    if i % 50 == 0:
        print(f"{i}/{len(timestamps)} Chunks geladen")
    time.sleep(0.2)

df_raw = pd.DataFrame(series, columns=["timestamp_ms", "pv_mw"])
df_raw.to_csv("../data/raw/smard_pv_realized_hour_2015-2026.csv", index=False)
print(f"Fertig: {len(df_raw)} Datenpunkte unter 'data/raw/smard_pv_realized_hour_2015-2026.csv' gespeichert")

0/598 Chunks geladen
50/598 Chunks geladen
100/598 Chunks geladen
150/598 Chunks geladen
200/598 Chunks geladen
250/598 Chunks geladen
300/598 Chunks geladen
350/598 Chunks geladen
400/598 Chunks geladen
450/598 Chunks geladen
500/598 Chunks geladen
550/598 Chunks geladen
Fertig: 100463 Datenpunkte unter 'data/raw/smard_pv_realized_hour_2015-2026.csv' gespeichert
